In [1]:
from datasets import load_dataset

ds_2024 = load_dataset("Reubencf/2024_events")
ds_2025 = load_dataset("Reubencf/2025_events")

from datasets import concatenate_datasets

combined_events = concatenate_datasets([
    ds_2024["train"],
    ds_2025["train"]
])

# convert section to lowercase because they are inconsistent across both datasets
events = combined_events.map(
    lambda x: {"section": x["section"].lower()}
)


In [4]:
print("\nColumns:")
print(events.column_names)
print("\nFeatures:")
print(events.features)

import random
from pprint import pprint

n_samples = 1

indices = random.sample(range(len(events)), n_samples)

print(f"\n{n_samples} Random events:")
for idx in indices:
    sample = events[idx]
    print(f"\nSample[{idx}]:")
    pprint(sample)

# count articles in each section
from collections import Counter

section_counts = Counter(events["section"])

print("\nSection counts:")
for section, count in section_counts.most_common():
    print(f"\t{section}: {count}")



Columns:
['month', 'year', 'day', 'section', 'content', 'instruction', 'response']

Features:
{'month': Value('string'), 'year': Value('int64'), 'day': Value('int64'), 'section': Value('string'), 'content': Value('string'), 'instruction': Value('string'), 'response': Value('string')}

1 Random events:

Sample[4829]:
{'content': 'Syrian civil war\n'
            'Israel and the Syrian civil war\n'
            'Israeli invasion of Syria (2024–present)\n'
            "Israeli tanks and troops enter Syria's Quneitra Governorate to "
            "reportedly establish a buffer zone, marking Israel's first "
            'crossing of the ceasefire line in 50 years. (The Times of Israel) '
            '(Al Jazeera)',
 'day': 8,
 'instruction': 'What were the immediate goals and implications of Israeli '
                "tanks and troops crossing the ceasefire line into Syria's "
                'Quneitra Governorate on December 8, 2024, a move reported to '
                'establish a buffer z

In [10]:
import requests, time

ollama_base_url = "http://localhost:11434/api/"
embedding_models = [
    "mxbai-embed-large",
    "nomic-embed-text"
]

for embedding_model in embedding_models:

    start = time.time()
    print(f"\n**** Embeddings using model {embedding_model}")
    response = requests.post(
        ollama_base_url + "embeddings",
        json={
            "model": embedding_model,
            "prompt": sample["instruction"]
        }
    )
    end = time.time()
    print(f"{len(response.json()["embedding"])} dimensions generated in {round(end-start, 3)}s")


**** Embeddings using model mxbai-embed-large
1024 dimensions generated in 0.953s

**** Embeddings using model nomic-embed-text
768 dimensions generated in 0.618s


In [11]:
llm_models = [
    "qwen3.5:9b",
    "llama3.2:3b",
    "llama3.2:1b",
    "gpt-oss:20b"
]


for llm_model in llm_models:
    start = time.time()

    print(f"\n\n*** Trial generate with {llm_model}:")
    response = requests.post(
        ollama_base_url + "generate",
        json={
            "model": llm_model,
            "prompt": f"Summarize the following text clearly and concisely:\n\n{sample["content"]}",
            "stream": False
        }
    )
    print("Summary: " + response.json()["response"])
    end = time.time()
    print(f"Summary Time: {round(end - start)}s") 

    start = time.time()
    response = requests.post(
        ollama_base_url + "generate",
        json={
            "model": llm_model,
            "prompt": f"Give three main concise and clear points from the text:\n\n{sample["content"]}",
            "stream": False
        }
    )
    print("Main points:\n" + response.json()["response"])

    end = time.time()
    print(f"Main points time: {round(end - start)}s") 



*** Trial generate with qwen3.5:9b:
Summary: Belarusian President Lukashenko has called for armed security patrols on streets and workplaces amid concerns over potential extremist crimes.
Summary Time: 19s
Main points:
1. Belarusian President Alexander Lukashenko is calling for armed security patrols.
2. These patrols are to be deployed on streets and within workplaces.
3. The request is justified by concerns over the possibility of "extremist" crimes.
Main points time: 22s


*** Trial generate with llama3.2:3b:
Summary: Belarusian President Alexander Lukashenko proposes implementing armed security patrols to prevent potential "extremist" crimes.
Summary Time: 2s
Main points:
Here are three main points from the text:

1. Belarusian President Alexander Lukashenko is calling for armed security patrols to be implemented on streets and in workplaces.
2. The reason cited by Lukashenko is the potential risk of "extremist" crimes.
3. Lukashenko's proposal aims to address concerns about extr